## 导入依赖

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from src.pipeline import ABSA_Qwen_CoT_Pipeline
from src.schemas import ABSAOutput

## 初始化 Pipeline

In [ ]:
try:
    # 這裡假設您已經在 src/pipeline.py 中實現了該類
    # 並且安裝了 bitsandbytes 用於 4bit 量化
    pipeline = ABSA_Qwen_CoT_Pipeline(
        model_name_or_path="Qwen/Qwen3-8B-Instruct", # 或本地路徑
        use_4bit=True
    )
    print("Pipeline 初始化成功！")
except ImportError:
    print("請確保安裝了 transformers, accelerate, bitsandbytes")
except Exception as e:
    print(f"初始化失敗: {e}")

## 单样本推理测试

In [ ]:
text_sample = "I loved the screen quality, it's amazing, but the battery life is terrible."

print(f"正在分析: {text_sample}")
result = pipeline.analyze(text_sample)

## 结果展示与 Pydantic 验证检查

In [ ]:
if result["error"]:
    print("分析過程中發生錯誤:", result["error"])
else:
    print("\n=== 思考過程 (CoT) ===")
    print(result["think_content"]) # 打印 <think> 標籤內的內容 [7]
    
    print("\n=== 最終結構化輸出 (JSON) ===")
    # 使用 Pydantic 的 model_dump 輸出字典
    analysis_obj = result["analysis"]
    print(analysis_obj.model_dump_json(indent=2))
    
    # 驗證提取的方面是否正確
    for aspect in analysis_obj.aspects:
        print(f"- 方面: {aspect.aspect_term}, 情感: {aspect.sentiment}")

## 批量测试(Optional)

In [ ]:
texts = []

for t in texts:
    print(f"\nAnalyzing: {t}")
    res = pipeline.analyze(t)
    if res["analysis"]:
        print(f"Detected: {len(res['analysis'].aspects)} aspects")